## Experiments with different models

**Import libraries**

In [11]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LinearRegression
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import mean_squared_error, r2_score
import joblib
from sklearn.metrics import precision_recall_curve, auc
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras.optimizers import Adam, SGD
from keras.models import load_model

2025-12-17 17:41:59.001165: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-12-17 17:41:59.290247: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:485] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-12-17 17:41:59.385631: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:8454] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-12-17 17:41:59.448984: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1452] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-12-17 17:41:59.630288: I tensorflow/core/platform/cpu_feature_guar

In [23]:
def prepare_data(data):
   # data['size'] = data['size'].astype(str)
   # data_dummies = pd.get_dummies(data, columns=['size'])
    # X = data_dummies.drop(['sample', 'regulator', 'target', 'interaction', 'size'], axis=1).values
    # y = data_dummies['interaction'].values
    X = data.drop(['sample', 'regulator', 'target', 'interaction', 'size'], axis=1).values
    y = data['interaction'].values
    X = np.asarray(X).astype(np.float32)
    y = np.asarray(y).astype(np.float32)
    return X, y

**Statistic**

In [28]:
training_set_10_1 = pd.read_csv(r'./caocao/toy dataset/training/toy_size10_1.csv', index_col=0)
training_set_10_1.info()

<class 'pandas.core.frame.DataFrame'>
Index: 450000 entries, 0 to 449999
Columns: 105 entries, size to t_step49
dtypes: int64(103), object(2)
memory usage: 363.9+ MB


In [3]:
def stat_size10():
    training_set_10_1 = pd.read_csv(r'./caocao/toy dataset/training/toy_size10_1.csv', index_col=0)
    training_set_10_2 = pd.read_csv(r'./caocao/toy dataset/training/toy_size10_2.csv', index_col=0)
    training_set_10_3 = pd.read_csv(r'./caocao/toy dataset/training/toy_size10_3.csv', index_col=0)
    training_set_10_4 = pd.read_csv(r'./caocao/toy dataset/training/toy_size10_4.csv', index_col=0)
    training_set_sum = pd.concat([training_set_10_1, training_set_10_2, training_set_10_3, training_set_10_4], ignore_index=True)
    valid_set = pd.read_csv(r'./caocao/toy dataset/validation/toy_size10_1.csv', index_col=0)
    test_set_report = pd.read_csv(r'./caocao/toy dataset/test/toy_size10_1.csv', index_col=0)
    train_true = len(training_set_sum[training_set_sum['interaction']==1])
    train_false = training_set_sum.shape[0] - train_true
    train_percent = round(train_true/train_false, 4)

    val_true = len(valid_set[valid_set['interaction']==1])
    val_false = valid_set.shape[0] - val_true
    val_percent = round(val_true/val_false, 4)

    test_true = len(test_set_report[test_set_report['interaction']==1])
    test_false = test_set_report.shape[0] - test_true
    test_percent = round(test_true/test_false, 4)

    return (training_set_sum.shape[0], train_true, train_false, train_percent), \
            (valid_set.shape[0], val_true, val_false, val_percent), (test_set_report.shape[0], test_true, test_false, test_percent)

In [5]:
def stat_size50():
    training_set_10_1 = pd.read_csv(r'./caocao/toy dataset/training/toy_size50_1.csv', index_col=0)
    training_set_10_2 = pd.read_csv(r'./caocao/toy dataset/training/toy_size50_2.csv', index_col=0)
    training_set_10_3 = pd.read_csv(r'./caocao/toy dataset/training/toy_size50_3.csv', index_col=0)
    training_set_10_4 = pd.read_csv(r'./caocao/toy dataset/training/toy_size50_4.csv', index_col=0)
    training_set_10_5 = pd.read_csv(r'./caocao/toy dataset/training/toy_size50_5.csv', index_col=0)
    training_set_sum = pd.concat([training_set_10_1, training_set_10_2, training_set_10_3, training_set_10_4, training_set_10_5], ignore_index=True)
    valid_set = pd.read_csv(r'./caocao/toy dataset/validation/toy_size50_1.csv', index_col=0)
    test_set_report = pd.read_csv(r'./caocao/toy dataset/test/toy_size50_1.csv', index_col=0)
    train_true = len(training_set_sum[training_set_sum['interaction']==1])
    train_false = training_set_sum.shape[0] - train_true
    train_percent = round(train_true/train_false, 4)

    val_true = len(valid_set[valid_set['interaction']==1])
    val_false = valid_set.shape[0] - val_true
    val_percent = round(val_true/val_false, 4)

    test_true = len(test_set_report[test_set_report['interaction']==1])
    test_false = test_set_report.shape[0] - test_true
    test_percent = round(test_true/test_false, 4)

    return (training_set_sum.shape[0], train_true, train_false, train_percent), \
            (valid_set.shape[0], val_true, val_false, val_percent), (test_set_report.shape[0], test_true, test_false, test_percent)

In [7]:
def stat_size100():
    '''
    Dataset contains all true size10, size50 and all size 100
    '''
    training_set_100_1 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_1.csv', index_col=0)
    training_set_100_2 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_2.csv', index_col=0)
    training_set_100_3 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_3.csv', index_col=0)
    training_set_100_4 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_4.csv', index_col=0)
    training_set_100_5 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_5.csv', index_col=0)
    training_set_100_6 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_6.csv', index_col=0)
    training_set_100_7 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_7.csv', index_col=0)
    training_set_100_8 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_8.csv', index_col=0)
    training_set_100_9 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_9.csv', index_col=0)
    training_set_100_10 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_10.csv', index_col=0)

    training_set_100_9_true = training_set_100_9[training_set_100_9['interaction']==1]
    training_set_100_10_true = training_set_100_10[training_set_100_10['interaction']==1]
    
    training_set_sum = pd.concat([training_set_100_1,
                                 training_set_100_2,training_set_100_3,training_set_100_4,
                                 training_set_100_5,training_set_100_6,training_set_100_7,training_set_100_8,
                                 training_set_100_9_true,training_set_100_10_true], ignore_index=True)
    del training_set_100_1
    del training_set_100_2
    del training_set_100_3
    del training_set_100_4
    del training_set_100_5
    del training_set_100_6
    del training_set_100_7
    del training_set_100_8
    del training_set_100_9
    del training_set_100_10
    valid_set = pd.read_csv(r'./caocao/toy dataset/validation/toy_size100_1.csv', index_col=0)  

    test_set_report = pd.read_csv(r'./caocao/toy dataset/test/toy_size100_1.csv', index_col=0)
    
    train_true = len(training_set_sum[training_set_sum['interaction']==1])
    train_false = training_set_sum.shape[0] - train_true
    train_percent = round(train_true/train_false, 4)

    val_true = len(valid_set[valid_set['interaction']==1])
    val_false = valid_set.shape[0] - val_true
    val_percent = round(val_true/val_false, 4)

    test_true = len(test_set_report[test_set_report['interaction']==1])
    test_false = test_set_report.shape[0] - test_true
    test_percent = round(test_true/test_false, 4)

    return (training_set_sum.shape[0], train_true, train_false, train_percent), \
            (valid_set.shape[0], val_true, val_false, val_percent), (test_set_report.shape[0], test_true, test_false, test_percent)

In [17]:
stat_size100()

((15930499, 451976, 15478523, 0.0292),
 (3960000, 90384, 3869616, 0.0234),
 (3960000, 90292, 3869708, 0.0233))

**Data preparation**

In [7]:
def load_size10():
    training_set_10_1 = pd.read_csv(r'./caocao/toy dataset/training/toy_size10_1.csv', index_col=0)
    training_set_10_2 = pd.read_csv(r'./caocao/toy dataset/training/toy_size10_2.csv', index_col=0)
    training_set_10_3 = pd.read_csv(r'./caocao/toy dataset/training/toy_size10_3.csv', index_col=0)
    training_set_10_4 = pd.read_csv(r'./caocao/toy dataset/training/toy_size10_4.csv', index_col=0)
    
    # training_set_50_1 = pd.read_csv(r'./caocao/toy dataset/training/toy_size50_1.csv', index_col=0)
    # training_set_50_2 = pd.read_csv(r'./caocao/toy dataset/training/toy_size50_2.csv', index_col=0)
    # training_set_50_3 = pd.read_csv(r'./caocao/toy dataset/training/toy_size50_3.csv', index_col=0)
    # training_set_50_4 = pd.read_csv(r'./caocao/toy dataset/training/toy_size50_4.csv', index_col=0)
    # training_set_50_5 = pd.read_csv(r'./caocao/toy dataset/training/toy_size50_5.csv', index_col=0)
    
    # training_set_50_1_true = training_set_50_1[training_set_50_1['interaction']==1]
    # training_set_50_2_true = training_set_50_2[training_set_50_2['interaction']==1]
    # training_set_50_3_true = training_set_50_3[training_set_50_3['interaction']==1]
    # training_set_50_4_true = training_set_50_4[training_set_50_4['interaction']==1]
    # training_set_50_5_true = training_set_50_5[training_set_50_5['interaction']==1]
    
    # training_set_sum = pd.concat([training_set_10_1, training_set_10_2, training_set_10_3, training_set_10_4,
    #                               training_set_50_1_true, training_set_50_2_true,
    #                              training_set_50_3_true, training_set_50_4_true, training_set_50_5_true], ignore_index=True)
    training_set_sum = pd.concat([training_set_10_1, training_set_10_2, training_set_10_3, training_set_10_4], ignore_index=True)
    valid_set = pd.read_csv(r'./caocao/toy dataset/validation/toy_size10_1.csv', index_col=0)
    X_train, y_train = prepare_data(training_set_sum)
    X_valid, y_valid = prepare_data(valid_set)
    return X_train, y_train, X_valid, y_valid

In [5]:
def load_size50():
    '''
    Dataset contains all size10, size50 and true positive of size 100
    '''
    # training_set_10_1 = pd.read_csv(r'./caocao/toy dataset/training/toy_size10_1.csv', index_col=0)
    # training_set_10_2 = pd.read_csv(r'./caocao/toy dataset/training/toy_size10_2.csv', index_col=0)
    # training_set_10_3 = pd.read_csv(r'./caocao/toy dataset/training/toy_size10_3.csv', index_col=0)
    # training_set_10_4 = pd.read_csv(r'./caocao/toy dataset/training/toy_size10_4.csv', index_col=0)

    training_set_50_1 = pd.read_csv(r'./caocao/toy dataset/training/toy_size50_1.csv', index_col=0)
    training_set_50_2 = pd.read_csv(r'./caocao/toy dataset/training/toy_size50_2.csv', index_col=0)
    training_set_50_3 = pd.read_csv(r'./caocao/toy dataset/training/toy_size50_3.csv', index_col=0)
    training_set_50_4 = pd.read_csv(r'./caocao/toy dataset/training/toy_size50_4.csv', index_col=0)
    training_set_50_5 = pd.read_csv(r'./caocao/toy dataset/training/toy_size50_5.csv', index_col=0)

    # training_set_100_1 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_1.csv', index_col=0)
    # training_set_100_2 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_2.csv', index_col=0)
    # training_set_100_3 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_3.csv', index_col=0)
    # training_set_100_4 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_4.csv', index_col=0)
    # training_set_100_5 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_5.csv', index_col=0)
    # training_set_100_6 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_6.csv', index_col=0)
    # training_set_100_7 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_7.csv', index_col=0)
    # training_set_100_8 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_8.csv', index_col=0)
    # training_set_100_9 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_9.csv', index_col=0)
    # training_set_100_10 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_10.csv', index_col=0)
    
    # training_set_100_1_true = training_set_100_1[training_set_100_1['interaction']==1]
    # training_set_100_2_true = training_set_100_2[training_set_100_2['interaction']==1]
    # training_set_100_3_true = training_set_100_3[training_set_100_3['interaction']==1]
    # training_set_100_4_true = training_set_100_4[training_set_100_4['interaction']==1]
    # training_set_100_5_true = training_set_100_5[training_set_100_5['interaction']==1]
    # training_set_100_6_true = training_set_100_6[training_set_100_6['interaction']==1]
    # training_set_100_7_true = training_set_100_7[training_set_100_7['interaction']==1]
    # training_set_100_8_true = training_set_100_8[training_set_100_8['interaction']==1]
    # training_set_100_9_true = training_set_100_9[training_set_100_9['interaction']==1]
    # training_set_100_10_true = training_set_100_10[training_set_100_10['interaction']==1]

    training_set_sum = pd.concat([training_set_50_1, training_set_50_2,training_set_50_3, training_set_50_4,training_set_50_5], ignore_index=True)
    # training_set_sum = pd.concat([training_set_10_1, training_set_10_2, training_set_10_3,training_set_10_4,
    #                              training_set_50_1, training_set_50_2,training_set_50_3, training_set_50_4,training_set_50_5,
    #                              training_set_100_1_true, training_set_100_2_true, training_set_100_3_true,
    #                              training_set_100_4_true, training_set_100_5_true, training_set_100_6_true,
    #                              training_set_100_7_true, training_set_100_8_true, training_set_100_9_true,
    #                              training_set_100_10_true], ignore_index=True)
    valid_set = pd.read_csv(r'./caocao/toy dataset/validation/toy_size50_1.csv', index_col=0)
    training_set_sum = training_set_sum.dropna()
    X_train, y_train = prepare_data(training_set_sum)
    X_valid, y_valid = prepare_data(valid_set)
    return X_train, y_train, X_valid, y_valid

In [5]:
def load_size100():
    '''
    Dataset contains all true size10, size50 and all size 100
    '''
    # training_set_10 = pd.read_csv(r'./caocao/toy dataset/training/toy_size10_merge.csv', index_col=0)
    # training_set_10_true = training_set_10[training_set_10['interaction']==1]

    # training_set_50 = pd.read_csv(r'./caocao/toy dataset/training/toy_size50_merge.csv', index_col=0)
    # training_set_50_true = training_set_50[training_set_50['interaction']==1]

    #training_set_100 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_merge.csv', index_col=0)
    training_set_100_1 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_1.csv', index_col=0)
    training_set_100_2 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_2.csv', index_col=0)
    training_set_100_3 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_3.csv', index_col=0)
    training_set_100_4 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_4.csv', index_col=0)
    training_set_100_5 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_5.csv', index_col=0)
    training_set_100_6 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_6.csv', index_col=0)
    training_set_100_7 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_7.csv', index_col=0)
    training_set_100_8 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_8.csv', index_col=0)
   # training_set_100_9 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_9.csv', index_col=0)
   # training_set_100_10 = pd.read_csv(r'./caocao/toy dataset/training/toy_size100_10.csv', index_col=0)

   # training_set_100_9_true = training_set_100_9[training_set_100_9['interaction']==1]
   # training_set_100_10_true = training_set_100_10[training_set_100_10['interaction']==1]
    
    # training_set_sum = pd.concat([training_set_10_true, training_set_50_true, training_set_100_1,
    #                              training_set_100_2,training_set_100_3,training_set_100_4,
    #                              training_set_100_5,training_set_100_6,training_set_100_7,training_set_100_8,
    #                              training_set_100_9_true,training_set_100_10_true], ignore_index=True)
    training_set_sum = pd.concat([training_set_100_1,training_set_100_2,training_set_100_3,training_set_100_4,
                                 training_set_100_5,training_set_100_6,training_set_100_7,training_set_100_8], ignore_index=True)
    del training_set_100_1
    del training_set_100_2
    del training_set_100_3
    del training_set_100_4
    del training_set_100_5
    del training_set_100_6
    del training_set_100_7
    del training_set_100_8
    
    valid_set = pd.read_csv(r'./caocao/toy dataset/validation/toy_size100_1.csv', index_col=0)    
    X_train, y_train = prepare_data(training_set_sum)
    X_valid, y_valid = prepare_data(valid_set)
    del valid_set
    return X_train, y_train, X_valid, y_valid

In [11]:
del training_set_100_1
del training_set_100_2
del training_set_100_3
del training_set_100_4
del training_set_100_5
del training_set_100_6
del training_set_100_7
del training_set_100_8
del training_set_100_9
del training_set_100_10

In [13]:
training_set_10 = pd.read_csv(r'./caocao/toy dataset/training/toy_size10_merge.csv', index_col=0)
training_set_10_true = training_set_10[training_set_10['interaction']==1]
del training_set_10

In [15]:
training_set_50 = pd.read_csv(r'./caocao/toy dataset/training/toy_size50_merge.csv', index_col=0)
training_set_50_true = training_set_50[training_set_50['interaction']==1]
del training_set_50

In [17]:
training_set_sum = pd.concat([training_set_10_true, training_set_50_true, training_set_sum], ignore_index=True)

In [19]:
valid_set = pd.read_csv(r'./caocao/toy dataset/validation/toy_size100_1.csv', index_col=0)

In [21]:
X_train, y_train = prepare_data(training_set_sum)
del training_set_sum

In [23]:
X_valid, y_valid = prepare_data(valid_set)
del valid_set

In [9]:
X_train, y_train, X_valid, y_valid = load_size10()

In [7]:
X_train, y_train, X_valid, y_valid = load_size50()

In [7]:
X_train, y_train, X_valid, y_valid = load_size100()

In [25]:
def get_report(model, report_filename, size, sample_number_max):
    with open(report_filename, 'w+') as f:
        test_set_report = pd.read_csv(fr'./caocao/toy dataset/test/toy_size{size}_1.csv', index_col=0)
        for i in range(1, sample_number_max+1):
            test_sample = test_set_report[test_set_report['sample']==i]
            X_test_sample, y_test_sample = prepare_data(test_sample)
            y_test_sample_pred = model.predict(X_test_sample)
            precision1, recall1, _ = precision_recall_curve(y_test_sample, y_test_sample_pred)
            aupr = auc(recall1, precision1)
            f.write(f'{aupr}\n')

In [11]:
def get_report_random(report_filename, size, sample_number_max):
    with open(report_filename, 'w+') as f:
        test_set = pd.read_csv(fr'./caocao/toy dataset/test/toy_size{size}_1.csv', index_col=0)
        for i in range(1, sample_number_max+1):
            test_sample = test_set[test_set['sample']==i]
            X_test_sample, y_test_sample = prepare_data(test_sample)
            aupr=0
            for sp in range(100):
                y_test_report_pred = np.random.rand(size*(size-1))
                precision1, recall1, _ = precision_recall_curve(y_test_sample, y_test_report_pred)
                aupr += auc(recall1, precision1)
            f.write(f'{aupr/100}\n')

In [35]:
get_report_random('./caocao/toy dataset/report/random_size100.txt', 100, 400)

In [84]:
test_set_report = pd.read_csv(fr'./caocao/validation/size10.csv', index_col=0)
test_set_report.head()

,size,sample,regulator,target,interaction,r_step0,r_step1,r_step2,r_step3,r_step4,...,t_step41,t_step42,t_step43,t_step44,t_step45,t_step46,t_step47,t_step48,t_step49,t_step50
0,10,0,betI,betA,1,0.406323,0.339645,0.319672,0.376442,0.247817,...,0.437771,0.335526,0.398282,0.406974,0.380937,0.345342,0.455723,0.486947,0.446095,0.469894
1,10,0,betI,betB,1,0.406323,0.339645,0.319672,0.376442,0.247817,...,0.348773,0.346059,0.338341,0.342637,0.411194,0.259389,0.430614,0.349655,0.394718,0.405301
2,10,0,betI,betT,1,0.406323,0.339645,0.319672,0.376442,0.247817,...,0.041398,0.025460,0.029818,0.013323,0.020699,0.024997,0.033653,0.029794,0.015124,0.014406
3,10,0,arcA,betI,1,0.728575,0.766482,0.641352,0.799297,0.645416,...,0.303531,0.278132,0.250648,0.320390,0.297727,0.319422,0.281178,0.324935,0.256094,0.308568
4,10,0,arcA,lldR,1,0.728575,0.766482,0.641352,0.799297,0.645416,...,0.355878,0.270155,0.211018,0.303878,0.280808,0.293172,0.269412,0.286260,0.318479,0.271814


In [40]:
test_set_report.info()

<class 'pandas.core.frame.DataFrame'>
Index: 180000 entries, 0 to 179999
Columns: 107 entries, size to t_step50
dtypes: float64(102), int64(3), object(2)
memory usage: 148.3+ MB


In [86]:
true_inte = test_set_report[test_set_report['interaction']==1]
true_inte.shape

(24300, 107)

In [88]:
false_inte = test_set_report[test_set_report['interaction']==0]
false_inte.shape

(155700, 107)

In [90]:
true_inte.shape[0]/false_inte.shape[0]

0.15606936416184972

### 0. MLP

In [19]:
X_train, y_train, X_valid, y_valid = load_size10()

In [11]:
model = keras.models.Sequential([ 
    keras.layers.Dense(1024, activation="relu", input_shape=X_train.shape[1:]),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(512, activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(256, activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(128, activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(64, activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(32, activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(16, activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(8, activation="relu"),
    keras.layers.BatchNormalization(),
    keras.layers.Dense(1, activation="sigmoid")
])

#optimizer = keras.optimizers.RMSprop(lr=0.001, rho=0.9)
optimizer = SGD(clipvalue=1.0, momentum=0.9, nesterov=True)
#optimizer=Adam(learning_rate=0.01, beta_1=0.9, beta_2=0.999)
early_stopping_cb = keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True)
model.compile(loss="mean_squared_error", optimizer=optimizer, metrics=['accuracy'])

/home/cao/anaconda3/lib/python3.12/site-packages/keras/src/layers/core/dense.py:87: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [13]:
history = model.fit(X_train, y_train, epochs=100, validation_data=(X_valid, y_valid), 
                    callbacks=[early_stopping_cb])

Epoch 1/100
 65870/497829 ━━━━━━━━━━━━━━━━━━━━ 1:08:58 10ms/step - accuracy: 0.9707 - loss: 0.0283

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



127435/497829 ━━━━━━━━━━━━━━━━━━━━ 59:08 10ms/step - accuracy: 0.9712 - loss: 0.0279

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



200753/497829 ━━━━━━━━━━━━━━━━━━━━ 47:26 10ms/step - accuracy: 0.9713 - loss: 0.0278

...

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



366330/497829 ━━━━━━━━━━━━━━━━━━━━ 20:58 10ms/step - accuracy: 0.9763 - loss: 0.0221

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



477316/497829 ━━━━━━━━━━━━━━━━━━━━ 3:16 10ms/step - accuracy: 0.9763 - loss: 0.0221

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



In [14]:
model.save('./caocao/toy dataset/report_across_size10/trained models/mlp_size100.h5')

In [16]:
model.save('./caocao/trained_model/model_50.h5')

In [146]:
model = load_model('./caocao/toy dataset/report_across_size10/trained models/mlp_size10.h5')

In [27]:
model = load_model('./caocao/toy dataset/report_across_size10/trained models/mlp_size100.h5')

In [ ]:
get_report(model, r'./caocao/toy dataset/singlesize/size50/mlp_size10.txt', 10, 2000)

3/3 ━━━━━━━━━━━━━━━━━━━━ 1s 175ms/step
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 5ms/step 
3/3 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step 
3/3 ━━━━━━━

In [30]:
get_report(model, r'./caocao/toy dataset/singlesize/size50/mlp_size50.txt', 50, 400)

77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
77/77 ━━━━━━━━━━━━━━━━━━━━ 0s 4ms/step
77/77 ━━━━━━━━━━━━━━━━━━━

In [31]:
get_report(model, r'./caocao/toy dataset/singlesize/size50/mlp_size100.txt', 100, 400)

310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
310/310 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step
310/310 ━━━━━━━━

### 1. Linear Regression

In [29]:
lin_reg = LinearRegression()
lin_reg.fit(X_train, y_train)

LinearRegression()

**Save trained model**

In [30]:
joblib.dump(lin_reg, './caocao/toy dataset/report_across_size10/trained models/lg_100.pkl')

['./caocao/toy dataset/report_across_size10/trained models/lg_100.pkl']

**Load saved model**

In [ ]:
lin_reg = joblib.load('linear_regression_model.pkl')

In [ ]:
lin_reg = joblib.load('./caocao/toy dataset/report_across_size10/trained models/lg_50.pkl')

**Evaluate model**

In [31]:
get_report(lin_reg, r'./caocao/toy dataset/report_across_size10/model100/lr_size10.txt', 10, 2000)

In [32]:
get_report(lin_reg, r'./caocao/toy dataset/report_across_size10/model100/lr_size50.txt', 50, 400)

In [33]:
get_report(lin_reg, r'./caocao/toy dataset/report_across_size10/model100/lr_size100.txt', 100, 400)

**ROC curve**

In [ ]:
thresholds = [i/100 for i in range(100, 0, -1)]
performances = []
for threshold in thresholds:
    predicted = get_round(y_test_pred, threshold)
    result = get_performace(test_set, predicted)
    performance = pd.DataFrame.from_dict(result, orient='index')
    performances.append(performance)
    performance.to_csv(fr'./caocao/others/Linear/size10_threshold_{threshold}.csv')

In [ ]:
curves_info = {i:[[],[]] for i in range(100)}
for per in performances:
    for i in range(100):
        curves_info[i][0].append(per.iloc[i, 3])
        curves_info[i][1].append(per.iloc[i, 4])

In [ ]:
import matplotlib.pyplot as plt
plt.figure(figsize=(8, 6))
plt.rcParams.update({
    'font.size': 12,              # Default font size for text
    'axes.titlesize': 20,         # Font size for axes titles
    'axes.labelsize': 15,         # Font size for x and y labels
    'xtick.labelsize': 12,        # Font size for x tick labels
    'ytick.labelsize': 12,        # Font size for y tick labels
    'legend.fontsize': 12,        # Font size for legend
    'figure.titlesize': 22        # Font size for figure title
})
plt.tick_params(axis='both',        # Apply changes to both x and y axes
                which='major',      # Apply changes to major ticks
                direction='inout',    # Ticks pointing outwards
                length=10,          # Length of ticks
                width=2           # Width of ticks
                #colors='red',       # Color of ticks
                #grid_color='black', # Color of gridlines
                #grid_alpha=0.5)     # Transparency of gridlines
               )
plt.xlim(0, 0.35)
plt.ylim(0, 0.8)
for k, v in curves_info.items():
    
    plt.plot(v[1], v[0], linewidth=3, marker='o', linestyle='dashed', color='#ff4f00')   
    plt.fill_between(v[1], v[0], color='#99ce3e', alpha=0.4)
    plt.text(0.2, 0.3, 'AUC', fontsize=14, color='#0d929a', ha='center', weight='bold')
    plt.text(0.07, 0.6, 'ROC', fontsize=14, color='#0d929a', ha='center',  weight='bold')
    plt.plot([0,v[1][-1]], [0, v[0][-1]], linewidth=3, linestyle='solid', color='#650000')
    plt.title(f'ROC curve - sample {k}')
    plt.xlabel('FP Rate')
    plt.ylabel('TP Rate')
    plt.savefig(fr'./caocao/others/Linear/ROC/s10_{k}.png')
    plt.clf()

### 2. SVM

In [79]:
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
svm_model = SVC(kernel='rbf', random_state=42)

# Train the model
svm_model.fit(X_train, y_train)

**Save trained model**

In [ ]:
joblib.dump(svm_model, 'svm_model_size50_add_10all_100true.pkl')

In [ ]:
joblib.dump(svm_model, './caocao/trained_model/svm_10.pkl')

**Load saved model**

In [ ]:
svm_model = joblib.load('svm_model.pkl')

**Evaluate model**

In [ ]:
# Predict on the validation set
y_val_pred = svm_model.predict(X_valid)

accuracy = accuracy_score(y_valid, y_val_pred)
print(f'Validation Accuracy: {accuracy}')

# More detailed performance metrics
print(classification_report(y_valid, y_val_pred))
print(confusion_matrix(y_valid, y_val_pred))

In [ ]:
# Predict on the test set
y_test_pred = svm_model.predict(X_test)

accuracy = accuracy_score(y_test, y_test_pred)
print(f'Test Accuracy: {accuracy}')

# More detailed performance metrics
print(classification_report(y_test, y_test_pred))
print(confusion_matrix(y_test, y_test_pred))

In [ ]:
get_report(svm_model, './caocao/report/svm_size10.txt', 10, 10)
get_report(svm_model, './caocao/report/svm_size50.txt', 50, 4)
get_report(svm_model, './caocao/report/svm_size100.txt', 100, 4)

In [ ]:
get_report(svm_model, './caocao/report_update/model10/svm_size50.txt', 50, 4)
get_report(svm_model, './caocao/report_update/model10/svm_size10.txt', 10, 10)
get_report(svm_model, './caocao/report_update/model10/svm_size100.txt', 100, 4)

### 3. Decision Tree

In [34]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [35]:
dt_model = DecisionTreeClassifier(random_state=42)

# Train the model
dt_model.fit(X_train, y_train)

DecisionTreeClassifier(random_state=42)

**Save trained model**

In [ ]:
joblib.dump(dt_model, 'decision_tree_model_size100_add_10true_50true.pkl')

In [36]:
joblib.dump(dt_model, './caocao/toy dataset/report_across_size10/trained models/dt_100.pkl')

['./caocao/toy dataset/report_across_size10/trained models/dt_100.pkl']

**Load saved model**

In [ ]:
# Load the model from the file
dt_model = joblib.load('decision_tree_model.pkl')

In [37]:
get_report(dt_model, r'./caocao/toy dataset/report_across_size10/model100/dt_size10.txt', 10, 2000)

In [38]:
get_report(dt_model, r'./caocao/toy dataset/report_across_size10/model100/dt_size50.txt', 50, 400)

In [39]:
get_report(dt_model, r'./caocao/toy dataset/report_across_size10/model100/dt_size100.txt', 100, 400)

**Evaluate model**

In [ ]:
# Predict on the validation set
y_val_pred = dt_model.predict(X_valid)

# Evaluate the model's performance
accuracy = accuracy_score(y_valid, y_val_pred)
print(f'Validation Accuracy: {accuracy}')

# More detailed performance metrics
print(classification_report(y_valid, y_val_pred))
print(confusion_matrix(y_valid, y_val_pred))

In [ ]:
get_report(dt_model, './caocao/report/dtree_size10.txt', 10, 10)
get_report(dt_model, './caocao/report/dtree_size50.txt', 50, 4)
get_report(dt_model, './caocao/report/dtree_size100.txt', 100, 4)

### 4. Random Forest

In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [15]:
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model
rf_model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

**Save trained model**

In [ ]:
joblib.dump(rf_model, 'random_forest_model_size100_add_10all_50true.pkl')

In [16]:
joblib.dump(rf_model, './caocao/toy dataset/report_across_size10/trained models/rf_100.pkl')

['./caocao/toy dataset/report_across_size10/trained models/rf_100.pkl']

**Load saved model**

In [ ]:
rf_model = joblib.load('random_forest_model.pkl')

**Evaluate model**

In [17]:
get_report(rf_model, r'./caocao/toy dataset/report_across_size10/model100/rf_size10.txt', 10, 2000)

In [18]:
get_report(rf_model, r'./caocao/toy dataset/report_across_size10/model100/rf_size50.txt', 50, 400)

In [19]:
get_report(rf_model, r'./caocao/toy dataset/report_across_size10/model100/rf_size100.txt', 100, 400)

### 5. K-Nearest Neighbors

In [20]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

**Feature Scaling (Important for KNN)**

In [22]:
knn_model = KNeighborsClassifier(n_neighbors=10)
# k=10 is the best
# Train the model
knn_model.fit(X_train, y_train)

KNeighborsClassifier(n_neighbors=10)

**Save trained model**

In [ ]:
joblib.dump(knn_model, 'knn_model_size100_add_10all_50true.pkl')

In [24]:
joblib.dump(knn_model, './caocao/toy dataset/report_across_size10/trained models/knn_100.pkl')

['./caocao/toy dataset/report_across_size10/trained models/knn_100.pkl']

In [35]:
joblib.dump(knn_model, './caocao/trained_model/toy_knn_100.pkl')

['./caocao/trained_model/toy_knn_100.pkl']

**Load saved model**

In [ ]:
knn_model = joblib.load('knn_model.pkl')

**Evaluate model**

In [26]:
get_report(knn_model, r'./caocao/toy dataset/report_across_size10/model100/knn_size10.txt', 10, 2000)

In [27]:
get_report(knn_model, r'./caocao/toy dataset/report_across_size10/model100/knn_size50.txt', 50, 400)

In [28]:
get_report(knn_model, r'./caocao/toy dataset/report_across_size10/model100/knn_size100.txt', 100, 400)

**Hyperparameter Tuning**

In [ ]:
# Try different values for n_neighbors
for k in range(1, 11):
    knn_model = KNeighborsClassifier(n_neighbors=k)
    knn_model.fit(X_train, y_train)
    y_val_pred = knn_model.predict(X_valid)
    accuracy = accuracy_score(y_valid, y_val_pred)
    print(f'Validation Accuracy with k={k}: {accuracy}')

### 6. XGBoost

In [ ]:
!pip install xgboost

In [9]:
import xgboost as xgb

**Convert data to DMatrix Format**

In [11]:
# Convert the datasets into DMatrix, which is the data structure that XGBoost uses
dtrain = xgb.DMatrix(X_train, label=y_train)
dval = xgb.DMatrix(X_valid, label=y_valid)

In [13]:
# Set up the parameters
params = {
    'max_depth': 3,         # Maximum depth of a tree
    'eta': 0.1,             # Learning rate
    'objective': 'binary:logistic',  # Binary classification objective
    'eval_metric': 'logloss' # Evaluation metric
}

# Optionally, you can add more parameters like:
# 'subsample': 0.8,  # Subsample ratio of the training instances
# 'colsample_bytree': 0.8,  # Subsample ratio of columns when constructing each tree

In [15]:
# Specify validation set for monitoring performance
evallist = [(dtrain, 'train'), (dval, 'eval')]

# Train the model
num_round = 100  # Number of boosting rounds
bst = xgb.train(params, dtrain, num_round, evals=evallist, early_stopping_rounds=10)

[0]	train-logloss:0.17225	eval-logloss:0.17225
[1]	train-logloss:0.16386	eval-logloss:0.16386
[2]	train-logloss:0.15651	eval-logloss:0.15651
[3]	train-logloss:0.15007	eval-logloss:0.15007
[4]	train-logloss:0.14444	eval-logloss:0.14444
[5]	train-logloss:0.13952	eval-logloss:0.13951
[6]	train-logloss:0.13522	eval-logloss:0.13521
[7]	train-logloss:0.13147	eval-logloss:0.13146
[8]	train-logloss:0.12821	eval-logloss:0.12820
[9]	train-logloss:0.12538	eval-logloss:0.12537
[10]	train-logloss:0.12292	eval-logloss:0.12291
[11]	train-logloss:0.12079	eval-logloss:0.12078
[12]	train-logloss:0.11896	eval-logloss:0.11895
[13]	train-logloss:0.11737	eval-logloss:0.11736
[14]	train-logloss:0.11601	eval-logloss:0.11600
[15]	train-logloss:0.11484	eval-logloss:0.11484
[16]	train-logloss:0.11384	eval-logloss:0.11383
[17]	train-logloss:0.11299	eval-logloss:0.11298
[18]	train-logloss:0.11227	eval-logloss:0.11226
[19]	train-logloss:0.11165	eval-logloss:0.11165
[20]	train-logloss:0.11113	eval-logloss:0.11112
[2

**Save trained model**

In [ ]:
bst.save_model('xgboost_model_size_add_10all_100true.json')

In [45]:
bst.save_model('./caocao/toy dataset/report_across_size10/trained models/xgb_100.json')

**Load saved model**

In [ ]:
loaded_bst = xgb.Booster()
loaded_bst.load_model('xgboost_model.json')

**Evaluate model**

In [17]:
def get_report_xgb(model, report_filename, size, sample_number_max):
    with open(report_filename, 'w+') as f:
        test_set = pd.read_csv(fr'./caocao/toy dataset/test/toy_size{size}_1.csv', index_col=0)
        for i in range(1, sample_number_max+1):
            test_sample = test_set[test_set['sample']==i]
            X_test_sample, y_test_sample = prepare_data(test_sample)
            dval = xgb.DMatrix(X_test_sample, label=y_test_sample)
            y_test_sample_pred = model.predict(dval)
            precision1, recall1, _ = precision_recall_curve(y_test_sample, y_test_sample_pred)
            aupr = auc(recall1, precision1)
            f.write(f'{aupr}\n')

In [49]:
get_report_xgb(bst, r'./caocao/toy dataset/report_across_size10/model100/xgb_size10.txt', 10, 2000)

In [24]:
get_report_xgb(bst, r'./caocao/toy dataset/report_across_size10/model100/xgb_size50.txt', 50, 400)

In [19]:
get_report_xgb(bst, r'./caocao/toy dataset/report_across_size10/model100/xgb_size100.txt', 100, 400)

### 7. LightGBM

In [ ]:
!pip install lightgbm

In [11]:
import lightgbm as lgb

**Convert data to LightGBM dataset format**

In [13]:
train_data = lgb.Dataset(X_train, label=y_train)
val_data = lgb.Dataset(X_valid, label=y_valid, reference=train_data)

In [15]:
# Set up the parameters
params = {
    'boosting_type': 'gbdt',  # Gradient Boosting Decision Tree
    'objective': 'binary',    # Binary classification
    'metric': 'binary_logloss', # Metric to evaluate
    'num_leaves': 31,         # Maximum tree leaves for base learners
    'learning_rate': 0.05,    # Learning rate
    'feature_fraction': 0.9   # Fraction of features to be used for each tree
}

# You can add more parameters like:
# 'bagging_fraction': 0.8,  # Subsample ratio of the training instances
# 'bagging_freq': 5,        # Frequency for bagging
# 'max_depth': -1,          # Maximum depth of the tree (unlimited if set to -1)

In [17]:
# Train the model
lgb_model = lgb.train(params, train_data, num_boost_round=100,\
                valid_sets=[train_data, val_data])

[LightGBM] [Info] Number of positive: 361477, number of negative: 15478523
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.977953 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 300
[LightGBM] [Info] Number of data points in the train set: 15840000, number of used features: 100
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.022821 -> initscore=-3.757010
[LightGBM] [Info] Start training from score -3.757010


**Save trained model**

In [19]:
# Save the model to a file
lgb_model.save_model('./caocao/toy dataset/report_across_size10/trained models/lgb_100.json')

**Load saved model**

In [ ]:
# Load the model from a file
loaded_bst = lgb.Booster(model_file='lightgbm_model.txt')

**Evaluate model**

In [21]:
def get_report_lgb(model, report_filename, size, sample_number_max):
    with open(report_filename, 'w+') as f:
        test_set = pd.read_csv(fr'./caocao/toy dataset/test/toy_size{size}_1.csv', index_col=0)
        for i in range(1, sample_number_max+1):
            test_sample = test_set[test_set['sample']==i]
            X_test_sample, y_test_sample = prepare_data(test_sample)
            y_test_sample_pred = model.predict(X_test_sample, num_iteration=model.best_iteration)
            precision1, recall1, _ = precision_recall_curve(y_test_sample, y_test_sample_pred)
            aupr = auc(recall1, precision1)
            f.write(f'{aupr}\n')

In [23]:
get_report_lgb(lgb_model, r'./caocao/toy dataset/report_across_size10/model100/lgb_size10.txt', 10, 2000)

In [25]:
get_report_lgb(lgb_model, r'./caocao/toy dataset/report_across_size10/model100/lgb_size50.txt', 50, 400)

In [27]:
get_report_lgb(lgb_model, r'./caocao/toy dataset/report_across_size10/model100/lgb_size100.txt', 100, 400)

### 8.Naive Bayes Classifier

In [29]:
nb_classifier = GaussianNB()
nb_classifier.fit(X_train, y_train)

GaussianNB()

In [31]:
get_report(nb_classifier, r'./caocao/toy dataset/report_across_size10/model100/nb_size10.txt', 10, 2000)

In [33]:
get_report(nb_classifier, r'./caocao/toy dataset/report_across_size10/model100/nb_size50.txt', 50, 400)

In [35]:
get_report(nb_classifier, r'./caocao/toy dataset/report_across_size10/model100/nb_size100.txt', 100, 400)

### 9. LSTM

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import LSTM, Dense, Dropout
from tensorflow.keras.preprocessing.sequence import pad_sequences
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [ ]:
# Normalize the feature data
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.fit_transform(X_val)